# Notebook 2 — Data Cleaning & Transformation

**Objetivo:** limpiar la capa Bronze y enriquecerla con columnas derivadas para producir la capa **Silver**.

| Paso | Descripción |
|------|-------------|
| 1 | Eliminación de precios inválidos |
| 2 | Extracción de componentes temporales |
| 3 | Categorización de precios |
| 4 | Decodificación de códigos |
| 5 | Limpieza de nulos en ciudades |
| 6 | Guardado capa Silver |

In [0]:
# ============================================
# NOTEBOOK 2: Cleaning & Transformation
# ============================================

from pyspark.sql.functions import col, year, month, quarter, when, count, avg
from pyspark.sql.functions import round as spark_round

print("="*70)
print("TRANSFORMACIÓN")
print("="*70)


In [0]:
# ============================================
# Cargar Bronze
# ============================================

df = spark.table("workspace.uk_housing.bronze_property_sales")

print(f" Registros: {df.count():,}")

## 1. Limpieza de precios
Se eliminan registros con `price` nulo, cero o negativo — valores imposibles en una transacción real.

In [0]:
# ============================================
# LIMPIEZA: Eliminar precios inválidos
# ============================================

print("\n" + "="*70)
print("LIMPIEZA DE PRECIOS")
print("="*70)

print(f"Registros antes: {df.count():,}")

# Eliminar precios nulos, negativos o cero
df_clean = df.filter(
    (col("price").isNotNull()) & 
    (col("price") > 0)
)

registros_eliminados = df.count() - df_clean.count()
print(f"Registros después: {df_clean.count():,}")
print(f"Registros eliminados: {registros_eliminados:,}")

# Ver estadísticas de precios limpios
print("\n--- Estadísticas de precios limpios ---")
display(df_clean.select("price").describe())

## 2. Componentes temporales
Se extraen `year`, `month` y `quarter` de `date_of_transfer` para facilitar agrupaciones en análisis posteriores.

In [0]:
# ============================================
# TRANSFORMACIÓN: Extraer componentes de fecha
# ============================================

print("\n" + "="*70)
print("EXTRACCIÓN DE COMPONENTES TEMPORALES")
print("="*70)

# Ya tenemos date_of_transfer como timestamp, solo extraemos componentes
df_clean = df_clean.withColumn("year", year("date_of_transfer")) \
                   .withColumn("month", month("date_of_transfer")) \
                   .withColumn("quarter", quarter("date_of_transfer"))

print("\n✓ Columnas temporales creadas: year, month, quarter")

# Verificar
display(df_clean.select("date_of_transfer", "year", "month", "quarter").limit(10))

# Ver distribución por año
print("\n--- Transacciones por año ---")
display(df_clean.groupBy("year").count().orderBy("year"))

## 3. Categorización de precios
Umbral basado en el mercado UK: Low (<200k), Medium (200k–500k), High (500k–1M), Premium (>1M).

In [0]:
# ============================================
# TRANSFORMACIÓN: Categorías de precio
# ============================================

print("\n" + "="*70)
print("CATEGORIZACIÓN DE PRECIOS")
print("="*70)

# Categorías basadas en el mercado UK
df_clean = df_clean.withColumn(
    "price_category",
    when(col("price") < 200000, "Low")
    .when((col("price") >= 200000) & (col("price") < 500000), "Medium")
    .when((col("price") >= 500000) & (col("price") < 1000000), "High")
    .otherwise("Premium")
)

print("\n--- Distribución por categoría de precio ---")
display(df_clean.groupBy("price_category") \
        .agg({"price": "count", "price": "avg"}) \
        .withColumnRenamed("count(price)", "total_transacciones") \
        .withColumnRenamed("avg(price)", "precio_promedio") \
        .orderBy("precio_promedio"))

## 4. Decodificación de códigos
Conversión de códigos a descripciones legibles: `property_type_desc`, `property_age` (new build vs resale) y `tenure_type` (freehold/leasehold).

In [0]:
# ============================================
# TRANSFORMACIÓN: Decodificar códigos
# ============================================

print("\n" + "="*70)
print("DECODIFICACIÓN DE CÓDIGOS")
print("="*70)

# Tipo de propiedad
df_clean = df_clean.withColumn(
    "property_type_desc",
    when(col("property_type") == "D", "Detached")
    .when(col("property_type") == "S", "Semi-Detached")
    .when(col("property_type") == "T", "Terraced")
    .when(col("property_type") == "F", "Flat/Apartment")
    .when(col("property_type") == "O", "Other")
    .otherwise("Unknown")
)

# Nueva construcción vs Reventa
df_clean = df_clean.withColumn(
    "property_age",
    when(col("old_new") == "Y", "New Build")
    .otherwise("Resale")
)

# Tipo de tenencia
df_clean = df_clean.withColumn(
    "tenure_type",
    when(col("duration") == "F", "Freehold")
    .when(col("duration") == "L", "Leasehold")
    .otherwise("Unknown")
)

print("\n✓ Códigos decodificados")

# Verificar
print("\n--- Distribución por tipo de propiedad ---")
display(df_clean.groupBy("property_type_desc").count().orderBy("count", ascending=False))

print("\n--- Nueva construcción vs Reventa ---")
display(df_clean.groupBy("property_age").count())

print("\n--- Freehold vs Leasehold ---")
display(df_clean.groupBy("tenure_type").count())

## 5. Nulos en ciudades
Los registros sin ciudad se rellenan con `'Unknown'` para evitar pérdida de datos en agrupaciones geográficas.

In [0]:
# ============================================
# LIMPIEZA: Valores nulos en ciudades
# ============================================

print("\n" + "="*70)
print("LIMPIEZA DE CIUDADES")
print("="*70)

# Contar nulos antes
nulos_antes = df_clean.filter(col("town_city").isNull()).count()
print(f"Ciudades nulas antes: {nulos_antes:,}")

# Rellenar con 'Unknown'
df_clean = df_clean.withColumn(
    "town_city",
    when(col("town_city").isNull(), "Unknown").otherwise(col("town_city"))
)

# Verificar
nulos_despues = df_clean.filter(col("town_city") == "Unknown").count()
print(f"Ciudades 'Unknown' después: {nulos_despues:,}")

## 6. Capa Silver
Se seleccionan 14 columnas finales y se persiste en `workspace.uk_housing.silver_property_sales` como tabla Delta.

In [0]:
# ============================================
# GUARDAR TABLA SILVER
# ============================================

print("\n" + "="*70)
print("CREACIÓN DE TABLA SILVER")
print("="*70)

spark.sql("DROP TABLE IF EXISTS silver_property_sales")

# Seleccionar columnas finales (must match schema exactly)
df_silver = df_clean.select(
    "transaction_id",
    "price",
    "date_of_transfer",
    "year",
    "month",
    "quarter",
    "price_category",
    "postcode",
    "property_type",
    "property_type_desc",
    "property_age",
    "tenure_type",
    "town_city",
    "district"
)

# Guardar como tabla Silver

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.uk_housing.silver_property_sales")

print("\n Tabla Silver creada: workspace.uk_housing.silver_property_sales")
print(f"✓ Total de registros: {df_silver.count():,}")

# Ver muestra
print("\nPrimeras 5 filas de Silver:")
display(df_silver.limit(5))